In [1]:
import torch
import torch.nn.functional as F

import pandas as pd

import preprocess
import update_model
import validate

In [2]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

DEVICE = get_device()
print('Using device:', str(DEVICE).upper(), "\n")

MODEL_DIR="./model"
DATA_PATH = "comb.txt"

TRAIN_DATA_BUCKET = "cbow-training-data-1f656"
MODEL_DATA_BUCKET = "cbow-model-data-1f656"

MPS is available
Using device: MPS 



In [3]:
# # === Run this code for the first model initialization ===

# # Download training data
# preprocess.download_data_from_gcs(
#     TRAIN_DATA_BUCKET,
#     DATA_PATH,
# )

# model, cfg = update_model.init_first_model(data_path=DATA_PATH)

/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Data already exists at data/comb.txt
 Initialising New CBOW-NS Model
   data path : data/comb.txt
   model dir : ./model
 -- Existing vocab size : 0
 -- New words added     : 18,525
 -- Merged vocab size   : 18,526
 -- Sequence length: 7,022,800 tokens  (saved → ./model/sequence.pt)
 -- Creating fresh CBOWModelNS (vocab=18,526, dim=256)

 -- Weights  saved → ./model/model_cpu.pt
 -- Config   saved → ./model/model_ns_config.json

 -- First-time initialisation done ✓


In [8]:
# === Run this code to initialize pretrained model ===

# Download training data
preprocess.download_data_from_gcs(
    TRAIN_DATA_BUCKET,
    DATA_PATH
)

# Download pretrained model and its configuration
preprocess.download_model_from_gcs(
    bucket_name=MODEL_DATA_BUCKET,
    download_dir=MODEL_DIR,
)

# Initialize pretrained model
model, cfg = update_model.init_model(
    new_data_path=f"data/{DATA_PATH}"
)

/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Data already exists at data/comb.txt


/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Latest version folder: version-3
Model already exists at ./model/model_cpu.pt
Model already exists at ./model/model_ns_config.json
Model already exists at ./model/sequence.pt
 CBOW Incremental Update
   new data : data/comb.txt
   device   : MPS
 -- Loaded weights from ./model/model_cpu.pt
 -- Skipping 'data/comb.txt': already ingested (hash match)
 -- Vocab unchanged; no layer resizing needed

 -- Done ✓


In [9]:
# === Train model ===
model = update_model.train(
    model=model,
    cfg=cfg,
    device=DEVICE,
    epochs=2,
)

# === Save model configuration locally ===
update_model.save(model, cfg)


 -- Training for 2 epoch(s) on 7,022,796 samples (NEG k=5, device=MPS)

   Epoch   1/2  loss=4935.8628  time=69.9s
   Epoch   2/2  loss=4612.4806  time=66.8s

 -- Weights  saved → ./model/model_cpu.pt
 -- Config   saved → ./model/model_ns_config.json


In [10]:
# # === Save model configuration in GCS and create new version ===
# preprocess.save_model_to_gcs(bucket_name=MODEL_DATA_BUCKET)

In [11]:
model = model.to(DEVICE)

word_embeddings = model.get_input_embeddings() # Shape: (vocab_size, embedding_dim)
word_embeddings_n = F.normalize(word_embeddings, p=2, dim=1)

window_size = cfg['window_size']
word2idx    = cfg["word2idx"]
idx2word    = {i: w for w, i in word2idx.items()}

In [19]:
print("\n" + "="*40)

test_word = "sweet"
top_k = 5
print(f"Top {top_k} similar words to '{test_word}':")
res = validate.find_similar_words(
    test_word, 
    word_embeddings_n, 
    word2idx, 
    idx2word,
    k=top_k
)

for w, score in res:
    print(f"  {w:15} {score:.4f}")
print("="*40)


Top 5 similar words to 'sweet':
  likeness        0.4610
  dearest         0.4451
  pupils          0.4360
  meek            0.4356
  friode          0.4345


In [26]:
word_1 = "yellow"
word_2 = "one"

v1 = word_embeddings[word2idx[word_1]]
v2 = word_embeddings[word2idx[word_2]]
print(f"Original: {F.cosine_similarity(v1, v2, dim=0).item()}")

Original: 0.0449836328625679


In [13]:
words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

In [21]:
embedding_size = cfg['embedding_shape'][1]
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

thresholds = [0.2, 0.4, 1, 3, 5, 6, 7]
# thresholds = [i for i in range(1, 11)]

for t in thresholds:
    
    # filtering
    B = (embeddings >= t).int()
    # C = torch.cov(B.T)
    C = B @ B.T

    print(f" -- threshold = {t}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_1-cov_matrix_{t}.csv")

 -- threshold = 0.2, nonzoer(B) = 1586
 -- threshold = 0.4, nonzoer(B) = 1370
 -- threshold = 1, nonzoer(B) = 787
 -- threshold = 3, nonzoer(B) = 40
 -- threshold = 5, nonzoer(B) = 0
 -- threshold = 6, nonzoer(B) = 0
 -- threshold = 7, nonzoer(B) = 0


In [22]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

deltas = [3, 2, 1, 0.5, 0.3, 0.1, 0.05]

for d in deltas:
    # filtering
    B = (torch.abs(embeddings) <= d).int()
    C = B @ B.T

    print(f"-- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_2-cov_matrix_{d}.csv")

-- delta = 3, nonzoer(B) = 3514
-- delta = 2, nonzoer(B) = 3179
-- delta = 1, nonzoer(B) = 2086
-- delta = 0.5, nonzoer(B) = 1113
-- delta = 0.3, nonzoer(B) = 692
-- delta = 0, nonzoer(B) = 0
-- delta = 1, nonzoer(B) = 2086
-- delta = 0.05, nonzoer(B) = 127
